In [4]:
# ============================================================
# CELL 1: INSTALL + IMPORT + CONFIG
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

!pip install -q datasets scikit-learn scipy joblib

# ============================================================
# IMPORTS
# ============================================================

import os
import sys
import csv
import json
import time
import random
import logging

from pathlib import Path

import numpy as np
import joblib
import scipy.sparse as sp

from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.preprocessing import normalize

# ============================================================
# SGD IMPORTS
# ============================================================

from sklearn.linear_model import (
    SGDClassifier,
    SGDRegressor
)

# ============================================================
# METRICS
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hinge_loss,
    mean_squared_error
)

from scipy.stats import (
    pearsonr,
    spearmanr
)

# ============================================================
# OUTPUT
# ============================================================

OUTPUT_ROOT = "/content/drive/MyDrive/STL_SVM_TFIDF_EARLYSTOP_RESULTS"

Path(OUTPUT_ROOT).mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# HP
# ============================================================

HP = dict(

    seed=42,

    ngram_max=3,

    max_features=200_000,

    alpha=1e-4,

    patience=3,

    max_epochs=20,
)

# ============================================================
# TASKS
# ============================================================

TASKS = {

    "sst2": {
        "hf": ("glue", "sst2"),
        "type": "cls",
        "a": "sentence",
        "b": None,
        "primary": "accuracy",
    },

    "qqp": {
        "hf": ("glue", "qqp"),
        "type": "cls",
        "a": "question1",
        "b": "question2",
        "primary": "accuracy",
    },

    "stsb": {
        "hf": ("glue", "stsb"),
        "type": "reg",
        "a": "sentence1",
        "b": "sentence2",
        "primary": "pearson",
    },
}

# ============================================================
# EXPERIMENTS
# ============================================================

EXPERIMENTS = [

    {
        "exp_name": "SVM-TFIDF-SST2",
        "model": "SVM+TFIDF",
        "task": "sst2",
    },

    {
        "exp_name": "SVM-TFIDF-QQP",
        "model": "SVM+TFIDF",
        "task": "qqp",
    },

    {
        "exp_name": "SVM-TFIDF-STSB",
        "model": "SVM+TFIDF",
        "task": "stsb",
    },
]

# ============================================================
# SEED
# ============================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

# ============================================================
# LOGGER
# ============================================================

def make_logger(name, log_path):

    Path(log_path).parent.mkdir(
        parents=True,
        exist_ok=True
    )

    lg = logging.getLogger(name)

    lg.setLevel(logging.INFO)

    lg.propagate = False

    for h in list(lg.handlers):
        lg.removeHandler(h)

    fmt = logging.Formatter(
        "[%(asctime)s] [%(levelname)s] %(message)s",
        "%Y-%m-%d %H:%M:%S"
    )

    fh = logging.FileHandler(
        log_path,
        mode="a",
        encoding="utf-8"
    )

    fh.setFormatter(fmt)

    sh = logging.StreamHandler(sys.stdout)

    sh.setFormatter(fmt)

    lg.addHandler(fh)
    lg.addHandler(sh)

    return lg

# ============================================================
# HISTORY WRITER
# ============================================================

class HistoryWriter:

    def __init__(self, out_dir):

        self.csv_path = Path(out_dir) / "history.csv"

        self.json_path = Path(out_dir) / "history.json"

        self.records = []

        self._fields = []

    def append(self, rec):

        for k in rec:

            if k not in self._fields:
                self._fields.append(k)

        self.records.append(rec)

        with open(
            self.csv_path,
            "w",
            newline="",
            encoding="utf-8"
        ) as f:

            w = csv.DictWriter(
                f,
                fieldnames=self._fields
            )

            w.writeheader()

            for r in self.records:

                w.writerow({
                    k: r.get(k, "")
                    for k in self._fields
                })

        self.json_path.write_text(
            json.dumps(
                self.records,
                indent=2,
                default=str
            ),
            encoding="utf-8"
        )

# ============================================================
# PAIR FEATURES
# ============================================================

def pair_features(
    Xa,
    Xb,
    add_cosine=True
):

    diff = (Xa - Xb).copy()

    diff.data = np.abs(diff.data)

    prod = Xa.multiply(Xb)

    feats = [
        Xa,
        Xb,
        diff,
        prod
    ]

    if add_cosine:

        an = normalize(Xa)

        bn = normalize(Xb)

        cos = np.asarray(
            an.multiply(bn).sum(axis=1)
        ).reshape(-1, 1)

        feats.append(
            sp.csr_matrix(cos)
        )

    return sp.hstack(feats).tocsr()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# ============================================================
# CELL 2: TRAIN + EARLY STOPPING + SAVE
# ============================================================

# ============================================================
# BENCHMARK
# ============================================================

def benchmark_sklearn(
    estimator,
    X_eval,
    n_warmup=1
):

    if X_eval.shape[0] >= 64:

        _ = estimator.predict(X_eval[:64])

    for _ in range(n_warmup):

        _ = estimator.predict(
            X_eval[:min(256, X_eval.shape[0])]
        )

    t0 = time.perf_counter()

    _ = estimator.predict(X_eval)

    el = time.perf_counter() - t0

    n = int(X_eval.shape[0])

    return {

        "inference_latency_ms":
            (el / max(n, 1)) * 1000.0,

        "throughput_samples_per_sec":
            n / max(el, 1e-9),

        "peak_vram_mb":
            0.0,

        "benchmark_samples":
            n,
    }

# ============================================================
# RUN ONE
# ============================================================

def run_one(exp, hp, root):

    out_dir = Path(root) / exp["exp_name"]

    out_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    lg = make_logger(
        exp["exp_name"],
        str(out_dir / "train.log")
    )

    lg.info(f"===== {exp['exp_name']} =====")

    set_seed(hp["seed"])

    task = exp["task"]

    meta = TASKS[task]

    primary = meta["primary"]

    is_pair = meta["b"] is not None

    is_reg = meta["type"] == "reg"

    # ========================================================
    # LOAD DATA
    # ========================================================

    raw = load_dataset(*meta["hf"])

    train_a = list(raw["train"][meta["a"]])

    val_a = list(raw["validation"][meta["a"]])

    train_b = (
        list(raw["train"][meta["b"]])
        if is_pair else None
    )

    val_b = (
        list(raw["validation"][meta["b"]])
        if is_pair else None
    )

    y_train = np.asarray(
        raw["train"]["label"],
        dtype=np.float32 if is_reg else np.int64
    )

    y_val = np.asarray(
        raw["validation"]["label"],
        dtype=np.float32 if is_reg else np.int64
    )

    lg.info(
        f"train={len(train_a):,} "
        f"val={len(val_a):,}"
    )

    # ========================================================
    # TF-IDF
    # ========================================================

    vec = TfidfVectorizer(
        ngram_range=(1, hp["ngram_max"]),
        max_features=hp["max_features"],
        sublinear_tf=True,
        lowercase=True,
        stop_words="english",
    )

    if is_pair:

        vec.fit(train_a + train_b)

        X_train = pair_features(
            vec.transform(train_a),
            vec.transform(train_b)
        )

        X_val = pair_features(
            vec.transform(val_a),
            vec.transform(val_b)
        )

    else:

        vec.fit(train_a)

        X_train = vec.transform(train_a)

        X_val = vec.transform(val_a)

    lg.info(
        f"X_train={X_train.shape} "
        f"X_val={X_val.shape}"
    )

    # ========================================================
    # MODEL
    # ========================================================

    if is_reg:

        clf = SGDRegressor(
            loss="squared_error",
            alpha=hp["alpha"],
            random_state=hp["seed"]
        )

    else:

        clf = SGDClassifier(
            loss="hinge",
            alpha=hp["alpha"],
            random_state=hp["seed"]
        )

    # ========================================================
    # TRAIN LOOP
    # ========================================================

    best_score = -float("inf")

    wait = 0

    hist = HistoryWriter(out_dir)

    for epoch in range(1, hp["max_epochs"] + 1):

        t0 = time.perf_counter()

        # ====================================================
        # TRAIN
        # ====================================================

        if is_reg:

            clf.partial_fit(
                X_train,
                y_train
            )

        else:

            clf.partial_fit(
                X_train,
                y_train,
                classes=np.unique(y_train)
            )

        train_time = time.perf_counter() - t0

        # ====================================================
        # PREDICT
        # ====================================================

        train_pred = clf.predict(X_train)

        val_pred = clf.predict(X_val)

        # ====================================================
        # METRICS
        # ====================================================

        if is_reg:

            train_loss = float(
                mean_squared_error(
                    y_train,
                    train_pred
                )
            )

            eval_loss = float(
                mean_squared_error(
                    y_val,
                    val_pred
                )
            )

            score = float(
                pearsonr(val_pred, y_val)[0]
            )

            metrics = {

                "accuracy": float("nan"),

                "precision": float("nan"),

                "recall": float("nan"),

                "macro_f1": float("nan"),

                "pearson": score,

                "spearman": float(
                    spearmanr(val_pred, y_val)[0]
                ),
            }

        else:

            train_loss = float(
                hinge_loss(
                    y_train,
                    clf.decision_function(X_train)
                )
            )

            eval_loss = float(
                hinge_loss(
                    y_val,
                    clf.decision_function(X_val)
                )
            )

            score = float(
                accuracy_score(
                    y_val,
                    val_pred
                )
            )

            metrics = {

                "accuracy": score,

                "precision": float(
                    precision_score(
                        y_val,
                        val_pred,
                        average="macro",
                        zero_division=0
                    )
                ),

                "recall": float(
                    recall_score(
                        y_val,
                        val_pred,
                        average="macro",
                        zero_division=0
                    )
                ),

                "macro_f1": float(
                    f1_score(
                        y_val,
                        val_pred,
                        average="macro",
                        zero_division=0
                    )
                ),

                "pearson": float("nan"),

                "spearman": float("nan"),
            }

        # ====================================================
        # BENCHMARK
        # ====================================================

        bn = benchmark_sklearn(
            clf,
            X_val
        )

        # ====================================================
        # SAVE HISTORY
        # ====================================================

        hist.append({

            "epoch": epoch,

            "train_loss": train_loss,

            "eval_loss": eval_loss,

            "accuracy": metrics["accuracy"],

            "precision": metrics["precision"],

            "recall": metrics["recall"],

            "macro_f1": metrics["macro_f1"],

            "pearson": metrics["pearson"],

            "spearman": metrics["spearman"],

            "inference_latency_ms":
                bn["inference_latency_ms"],

            "throughput_samples_per_sec":
                bn["throughput_samples_per_sec"],

            "peak_vram_mb":
                bn["peak_vram_mb"],

            "benchmark_samples":
                bn["benchmark_samples"],

            "time_per_epoch":
                train_time,
        })

        lg.info(
            f"Epoch {epoch} | "
            f"score={score:.4f}"
        )

        # ====================================================
        # EARLY STOPPING
        # ====================================================

        if score > best_score:

            best_score = score

            wait = 0

            joblib.dump(
                {
                    "vectorizer": vec,
                    "model": clf,
                    "task": task,
                    "is_pair": is_pair,
                },
                out_dir / "best_model.joblib"
            )

            lg.info("NEW BEST")

        else:

            wait += 1

            lg.info(
                f"No improvement "
                f"{wait}/{hp['patience']}"
            )

        if wait >= hp["patience"]:

            lg.info("EARLY STOPPING")

            break

    lg.info("DONE")

# ============================================================
# RUN ALL
# ============================================================

def run_all():

    for e in EXPERIMENTS:

        try:

            run_one(
                e,
                HP,
                OUTPUT_ROOT
            )

        except Exception as ex:

            print(
                f"FAILED: {e['exp_name']}"
            )

            print(ex)

# ============================================================
# MAIN
# ============================================================

run_all()

[2026-05-18 03:41:15] [INFO] ===== SVM-TFIDF-SST2 =====
[2026-05-18 03:41:25] [INFO] train=67,349 val=872
[2026-05-18 03:41:31] [INFO] X_train=(67349, 109632) X_val=(872, 109632)
[2026-05-18 03:41:31] [INFO] Epoch 1 | score=0.7970
[2026-05-18 03:41:34] [INFO] NEW BEST
[2026-05-18 03:41:34] [INFO] Epoch 2 | score=0.7947
[2026-05-18 03:41:34] [INFO] No improvement 1/3
[2026-05-18 03:41:34] [INFO] Epoch 3 | score=0.7947
[2026-05-18 03:41:34] [INFO] No improvement 2/3
[2026-05-18 03:41:34] [INFO] Epoch 4 | score=0.7936
[2026-05-18 03:41:35] [INFO] No improvement 3/3
[2026-05-18 03:41:35] [INFO] EARLY STOPPING
[2026-05-18 03:41:35] [INFO] DONE
[2026-05-18 03:41:35] [INFO] ===== SVM-TFIDF-QQP =====
[2026-05-18 03:41:58] [INFO] train=363,846 val=40,430
[2026-05-18 03:42:37] [INFO] X_train=(363846, 800001) X_val=(40430, 800001)
[2026-05-18 03:42:38] [INFO] Epoch 1 | score=0.7685
[2026-05-18 03:42:41] [INFO] NEW BEST
[2026-05-18 03:42:41] [INFO] Epoch 2 | score=0.7674
[2026-05-18 03:42:41] [INF